# Linked Lists

Singly and doubly linked lists: building nodes, traversal, insertion/deletion, and classic interview exercises (reversal, cycle detection, merging).

## Linked lists — singly vs. doubly, and why they're not made obsolete by ordered dicts

A **linked list** stores elements as a chain of nodes:

- **Singly linked list**: each node holds a `value` and a `next` pointer to the following node. Traversal is one-directional only — to remove a node you need a reference to its *previous* node (to re-point its `next`), which you can't get by going backward.
- **Doubly linked list**: each node also holds a `prev` pointer. You can traverse both directions, and — crucially — remove a node given only a reference to *that node itself* (no need to separately track its predecessor).

**vs. dicts with preserved insertion order (Python 3.7+):** a dict can cheaply push to the "just inserted" end (`del d[key]; d[key] = value`) and cheaply pop the oldest entry (`next(iter(d))` + `del`). What it **can't** do cheaply is move an existing item to the *front*, or splice a new node into the *middle* — dict's ordering only supports operating on one end. A doubly linked list gives O(1) insert/remove **anywhere**, given a node reference — at either end or in the middle — which is the general, language-agnostic guarantee dict's insertion-order trick doesn't provide.

**Main drawback: no random access.** Unlike an array, there's no indexing into memory — finding/accessing the k-th element, or searching for a value, means walking the chain node by node from the head. So lookup/search is O(n) worst case, versus O(1) by index for arrays or O(1) average by key for dicts. Linked lists trade lookup speed for cheap structural editing.

**Best-suited application: caches.** An LRU cache is the canonical use case — combining a dict (O(1) key → node lookup) with a doubly linked list (O(1) move-to-front on access, O(1) evict-from-the-back when full). The cache never needs to search or index into the list by position; it only ever touches nodes it already has a direct reference to (via the dict), which is exactly what linked lists are good at and dicts alone aren't (arbitrary reordering, not just push-to-one-end).

In [ ]:
# TODO: Define the building block: a Node class.
#
# Singly linked node: holds a value and a pointer to the next node.
# Doubly linked node: holds a value, a pointer to the next node, AND a
# pointer to the previous node.
#
# Start with SinglyNode, then DoublyNode.

'''
this is my naive first attempt implementation;
have an llm check the code after running simple tests so there might small edge cases missed :(
'''

class SinglyNode:
    def __init__(self, value):
        self.value = value
        self.next = None

class SinglyLinkedList:
    def __init__(self):
        self._head = None
        self._tail = None
        self._length = 0
    
    def append(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._tail
            self._tail = SinglyNode(value)
            _current.next = self._tail
        
        self._length += 1
    

    def prepend(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._head
            self._head = SinglyNode(value)
            self._head.next = _current
        
        self._length += 1
    
    def delete(self, value):

        if not self._head:
            raise ValueError("this list is empty!!")

        elif not self._head.next:
            if self._head.value == value:
                self._length -= 1
                self._head = None
                self._tail = None
        
        elif self._head.value == value:
            self._head = self._head.next
            self._length -= 1
        
        else:
            prev_node = self._head
            curr_node = self._head.next

            while prev_node.next:

                if curr_node.value == value:
                    prev_node.next = curr_node.next
                    self._length -= 1
                    if not prev_node.next:
                        self._tail = prev_node
                    break

                prev_node = curr_node
                curr_node = curr_node.next
                
    def insert(self, index, value):
        if index > self._length or index < 0:
            raise ValueError("out of bound!!")

        if index == 0: 
            new_node = SinglyNode(value)
            new_node.next = self._head
            self._head = new_node
        else:
            node = self._head 
            for i in range(index - 1):
                node = node.next     
            new_node = SinglyNode(value)
            temp = node.next
            node.next = new_node
            new_node.next = temp

        if not new_node.next:
            self._tail = new_node
    
        self._length += 1
    
    def to_list(self):
        node = self._head
        values = []
        for i in range(self._length):
            values.append(node.value)
            node = node.next
        return values
    
    
            
lst = SinglyLinkedList()
#lst.prepend(1)
#lst.prepend(2)
#lst.append(2)
#lst.append(2)
#lst.append(3)
#lst.append(4)
#print(lst.to_list())
lst.insert(0, 7)
lst.append(2)
#print(lst.to_list())
#lst.delete(1)
print(lst.to_list())

[7, 2]


In [86]:
# TODO: Define the building block: a Node class.
#
# Singly linked node: holds a value and a pointer to the next node.
# Doubly linked node: holds a value, a pointer to the next node, AND a
# pointer to the previous node.
#
# Start with SinglyNode, then DoublyNode.

## this is closer to the most common implementation

class SinglyNode:
    def __init__(self, value):
        self.value = value
        self.next = None

class SinglyLinkedList:
    def __init__(self):
        self._head = None
        self._tail = None
        self._length = 0
    
    def append(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._tail
            self._tail = SinglyNode(value)
            _current.next = self._tail
        
        self._length += 1
    

    def prepend(self, value):
        if self._length == 0:
            _current = SinglyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._head
            self._head = SinglyNode(value)
            self._head.next = _current
        
        self._length += 1
    
    def delete_at(self, index):
        
        if index >= self._length or index <0: 
            raise ValueError("index value is out of bound")

        dummy = SinglyNode(None)
        dummy.next = self._head
        node = dummy

        for i in range(index):
            node = node.next
        
        node.next = node.next.next
        self._length -= 1
        self._head = dummy.next
        if not node.next:
            self._tail = node if node is not dummy else None

    
    def delete(self, value):
        if not self._head:
            raise ValueError("this list is empty!!")
        
        dummy = SinglyNode(None)
        dummy.next = self._head

        prev_node = dummy
        curr_node = dummy.next

        while prev_node.next:
            if curr_node.value == value:
                prev_node.next = curr_node.next
                self._length -= 1
                if not prev_node.next:
                    self._tail = prev_node if prev_node is not dummy else None
                break

            prev_node = curr_node
            curr_node = curr_node.next
        
        self._head = dummy.next
    
    def reverse(self):
        if not self._head:
            raise ValueError("this list is empty!!")

        dummy = SinglyNode(None)
        dummy.next = self._head

        prev_node = dummy
        curr_node = dummy.next
       
        while curr_node:
            temp = curr_node.next 
            curr_node.next = prev_node if prev_node is not dummy else None
            prev_node = curr_node 
            curr_node = temp
        
        self._head = self._tail
        self._tail = dummy.next
            
                
    def insert(self, index, value):
        if index > self._length or index <0: 
            raise ValueError("index value is out of bound")

        dummy = SinglyNode(None)
        dummy.next = self._head
        node = dummy

        for i in range(index):
            node = node.next

        new_node = SinglyNode(value)
        temp = node.next
        node.next = new_node
        new_node.next = temp
        self._length += 1

        self._head = dummy.next
        if not new_node.next:
            self._tail = new_node
            
    def to_list(self):
        node = self._head
        if not self._head:
            return []
            
        values = []
        while node.next:
            values.append(node.value)
            node = node.next
        values.append(node.value)
        return values
    

            
lst = SinglyLinkedList()
lst.prepend(1)
lst.append(2)
lst.append(3)
lst.prepend(2)
print(lst.to_list())
#lst.insert(0, 7)
#print(lst.to_list())
#lst.delete_at(3)
lst.reverse()
print(lst.to_list())
print("head:", lst._head.value, lst._head.next)
print("tail:", lst._tail.value, lst._tail.next)

[2, 1, 2, 3]
[3, 2, 1, 2]
head: 3 <__main__.SinglyNode object at 0x00000185C47BB0B0>
tail: 2 None


## `__setattr__` vs `__setitem__` — and why Python doesn't need Java-style setters

- **`__setattr__`** is the hook behind dot-assignment: `obj.attr = value`. Every Python object automatically inherits a working `__setattr__` from the base `object` class — you never have to write it yourself. That's exactly why `node.next = new_node` (see `SinglyLinkedList.append`/`prepend`) just works with no method defined for it.
- **`__setitem__`** is the hook behind bracket-assignment: `obj[key] = value`. Unlike `__setattr__`, this is **not** provided by default — you must explicitly define it yourself if you want that syntax to work (which is exactly what we did for `SimpleDict`).

**vs. Java:** Java has no equivalent hook mechanism at all. `obj.field = value` only works there because the field happens to be visible (public/package-private) — it's plain field access, not something intercepted by the language. Java's convention of writing explicit `setField(...)` methods is a design discipline (encapsulation — so you *could* add validation or change internals later without breaking callers), not a technical requirement the way `__setitem__` is in Python. Python gets the "just assign it" ergonomics for free via `__setattr__`, and if you ever need custom validation/behavior later, you can add it (via `@property`/`.setter` or overriding `__setattr__`) without changing how callers write `obj.attr = value`.

In [87]:
# TODO: Define the building block: a Node class.
#
# Singly linked node: holds a value and a pointer to the next node.
# Doubly linked node: holds a value, a pointer to the next node, AND a
# pointer to the previous node.
#
# Start with SinglyNode, then DoublyNode.

### this is a simple implmentation of DoublyList

class DoublyNode:
    def __init__(self, value):
        self.value = value
        self.next = None
        self.prev = None

class DoublyLinkedList:
    def __init__(self):
        self._head = None
        self._tail = None
        self._length = 0
    
    def append(self, value):
        if self._length == 0:
            _current = DoublyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._tail
            self._tail = DoublyNode(value)
            _current.next = self._tail
            self._tail.prev = _current
        
        self._length += 1
    

    def prepend(self, value):
        if self._length == 0:
            _current = DoublyNode(value)
            self._head = _current
            self._tail = _current
            
        else:
            _current = self._head
            self._head = DoublyNode(value)
            self._head.next = _current
            _current.prev = self._head
        
        self._length += 1

    
    def to_list(self):
        node = self._head
        if not self._head:
            return []
            
        values = []
        while node.next:
            values.append(node.value)
            node = node.next
        values.append(node.value)
        return values
    
    def to_list_reverse(self):
        node = self._tail
        values = []
        for i in range(self._length):
            values.append(node.value)
            node = node.prev
        return values

            
lst = DoublyLinkedList()
lst.prepend(1)
lst.append(2)
lst.append(3)
lst.append(4)
lst.to_list_reverse()

[4, 3, 2, 1]

## Why "O(1) insert/delete" needs an asterisk

Adding or removing a node has two separate costs, and only one of them is actually O(1):

1. **Finding the spot.** A linked list has no random access — there's no way to jump straight to "the k-th node" or "the node holding value X" the way an array indexes into memory. The only way to get there is to walk the chain link by link, starting from `head` (or `tail`). That walk is O(n) in the worst case.
2. **Splicing at a spot you're already at.** Once you're holding a reference to the right node(s), rewiring `next`/`prev` pointers to insert or remove is O(1) — a fixed, small number of pointer reassignments, no matter how long the list is.

So the real rule is: **splicing is O(1); finding is O(n).** What determines the overall cost of an operation is whether the position is already known or has to be searched for:

- `append` / `prepend`: O(1) overall — the list already tracks `head` and `tail` directly, so there's nothing to search for.
- "Insert at index k" / "delete the node with value X": O(n) overall — O(n) to walk to that spot, plus O(1) to splice once there. The search dominates.
- True O(1) insert/delete at an *arbitrary* spot only happens when something else already handed you a direct node reference, skipping the search entirely — e.g., a dict mapping key → node, as in an LRU cache.